In [ ]:
import requests, pandas as pd, time
from datetime import datetime, timezone
BASE = "https://api.elections.kalshi.com/trade-api/v2"

def get(path, **p):
    time.sleep(0.1)  # stay under rate limits
    return requests.get(BASE + path, params={k: v for k, v in p.items() if v is not None}).json()

s = pd.json_normalize(get("/series", category="Sports")["series"])
print(s[s.title.str.contains("NBA|ATP|WTA", case=False)][["ticker", "title"]])

                  ticker                    title
0     KXNBASPORTSMANSHIP  NBA Sportsmanship Award
3         KXWTASETWINNER           WTA Set Winner
13        KXNBAWINRECORD       OKC NBA Win Record
22           KXATPANYSET       ATP Any Set Winner
25        KXATPS1GWINNER    ATP set 1 game winner
...                  ...                      ...
3797       KXCOACHOUTNBA            NBA Coach Out
3828       KXNBA4QWINNER   NBA 4th Quarter Winner
3843            KXNBAROY   NBA Rookie of the Year
3865      KXNBAWINMARGIN           NBA Win Margin
3878      KXLEADERNBAREB           NBA RPG Leader

[243 rows x 2 columns]


In [ ]:
pd.set_option("display.max_rows", None)
print(s[s.title.str.contains("ATP", case=False)][["ticker", "title"]])


                      ticker  \
22               KXATPANYSET   
25            KXATPS1GWINNER   
71              KXATPSSPREAD   
98            KXATPGAMETOTAL   
187           KXATPS5GWINNER   
228             KXATPCOMPETE   
313                KXATPWDDF   
411                KXATPACES   
490             KXATPDOUBLES   
758          KXATPEXACTMATCH   
897                  KXATPIT   
959              KXATPFINALS   
971           KXATPS2GWINNER   
1066          KXATPS4GWINNER   
1132              KXATPMATCH   
1176          KXATPS3GWINNER   
1371            KXATPNEXTGEN   
1485          KXATPGRANDSLAM   
1593             KXATPGTOTAL   
1599                KXATPMIA   
1625  KXATPCHALLENGERDOUBLES   
1631       KXCHALLENGERMATCH   
1642         KXATPGAMESPREAD   
1651              KXATP1RANK   
1727                KXMCMMEN   
1763                KXATPMEN   
1900            KXATPGSPREAD   
1933     KXATPGRANDSLAMFIELD   
2039               KXATPGAME   
2398            KXATPGWINNER   
2462    

In [ ]:
def settled(series, n=None, **p):
    out, cur = [], None
    while True:
        r = get("/markets", series_ticker=series, status="settled", limit=1000, cursor=cur, **p)
        out += r.get("markets", []); cur = r.get("cursor")
        if not cur or (n and len(out) >= n): return out[:n] if n else out

def candles(series, m, hours, interval):
    end = int(datetime.fromisoformat(m["close_time"].replace("Z", "+00:00")).timestamp())
    c = get(f"/series/{series}/markets/{m['ticker']}/candlesticks",
            start_ts=end - hours * 3600, end_ts=end, period_interval=interval)
    return [{"ticker": m["ticker"], "title": m["title"], "result": m.get("result"), **k}
            for k in c.get("candlesticks", [])]

In [ ]:
for tour in ["KXATPMATCH"]:
    ms = settled(tour, n=400)
    pd.json_normalize([r for m in ms for r in candles(tour, m, 8, 1)]).to_csv(f"{tour}.csv", index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
codes = set(pd.read_csv("KXATPMATCH.csv").ticker.str.split("-").str[1])
sw = [m for m in settled("KXATPSETWINNER", n=3000) if m["ticker"].split("-")[1] in codes]
print(len(sw), [m["ticker"] for m in sw[:5]])
pd.json_normalize(sw).to_csv("setwinner_markets.csv", index=False)
pd.json_normalize([r for m in sw for r in candles("KXATPSETWINNER", m, 8, 1)]).to_csv("setwinner_candles.csv", index=False)

1060 ['KXATPSETWINNER-26SEP28CUIALT-2-CUI', 'KXATPSETWINNER-26SEP28CUIALT-2-ALT', 'KXATPSETWINNER-26SEP28CUIALT-1-CUI', 'KXATPSETWINNER-26SEP28CUIALT-1-ALT', 'KXATPSETWINNER-26SEP28MEDSAF-2-SAF']


In [9]:
r = get("/markets", series_ticker="KXATPS1GWINNER", limit=3)
print(len(r.get("markets", [])), [m["ticker"] for m in r.get("markets", [])])
h = get("/historical/markets", series_ticker="KXATPS1GWINNER", limit=3)
print(len(h.get("markets", [])), [m["ticker"] for m in h.get("markets", [])])
print(get("/events", series_ticker="KXATPS1GWINNER", limit=2))

0 []
3 ['KXATPS1GWINNER-26JUN06SHISAKG10-SHI', 'KXATPS1GWINNER-26JUN06SHISAKG10-SAK', 'KXATPS1GWINNER-26JUN06VISYMEG10-YME']
{'cursor': 'CgYIwOKQ0QYSHktYQVRQUzFHV0lOTkVSLTI2SlVOMDZIU1VHQUxHNQ', 'events': [{'category': 'Sports', 'collateral_return_type': 'MECNET', 'event_ticker': 'KXATPS1GWINNER-26JUN06HSUGALG6', 'exchange_index': 0, 'last_updated_ts': '2026-08-28T20:54:59.498028Z', 'mutually_exclusive': True, 'product_metadata': {'competition': 'ATP Stuttgart', 'competition_scope': 'Game 6'}, 'series_ticker': 'KXATPS1GWINNER', 'settlement_sources': [{'name': 'ATP', 'url': 'https://www.atptour.com/'}], 'strike_period': '', 'sub_title': 'Yu Hsiou Hsu vs Alexis Galarneau Set 1 Game 6 (Jun 6)', 'title': 'Yu Hsiou Hsu vs Alexis Galarneau: Set 1 Game 6'}, {'category': 'Sports', 'collateral_return_type': 'MECNET', 'event_ticker': 'KXATPS1GWINNER-26JUN06HSUGALG5', 'exchange_index': 0, 'last_updated_ts': '2026-08-28T20:54:59.502443Z', 'mutually_exclusive': True, 'product_metadata': {'competitio

In [10]:
import re
g = all_mk("KXATPS1GWINNER", hist=True, n=40000)
print(len(g), "game markets |", pd.Series([m.get("result") for m in g]).value_counts().to_dict(), "| oldest:", g[-1]["ticker"])
gw = pd.DataFrame([{"code": re.sub(r"G\d+$", "", m["ticker"].split("-")[1]), "winner": m.get("yes_sub_title")}
                   for m in g if m.get("result") == "yes"])
if len(gw): gw.groupby(["code", "winner"]).size().rename("s1_games").reset_index().to_csv("set1_games.csv", index=False)
print(len(gw), "game wins saved")

246 game markets | {'yes': 105, 'no': 105, 'scalar': 36} | oldest: KXATPS1GWINNER-26JUN05MENZVEG1-MEN
105 game wins saved


In [11]:
import numpy as np
def all_mk(series, hist=False, n=6000):
    out, cur = [], None
    while True:
        r = get("/historical/markets" if hist else "/markets", series_ticker=series, limit=1000,
                cursor=cur, status=None if hist else "settled")
        out += r.get("markets", []); cur = r.get("cursor")
        if not cur or len(out) >= n: return out

def cdl(tk, a, b):
    for path in [f"/series/KXATPMATCH/markets/{tk}/candlesticks", f"/historical/markets/{tk}/candlesticks"]:
        c = get(path, start_ts=a, end_ts=b, period_interval=1).get("candlesticks")
        if c: return c
    return []

f = lambda c, k: float(c[k]["close_dollars"]) if c.get(k) and c[k].get("close_dollars") is not None else np.nan
match = {m["ticker"]: m for m in all_mk("KXATPMATCH") + all_mk("KXATPMATCH", hist=True, n=6000)}
s1 = [m for m in all_mk("KXATPSETWINNER") + all_mk("KXATPSETWINNER", hist=True, n=15000)
      if m["ticker"].split("-")[2] == "1" and m.get("result") == "yes"]
print(len(match), "match markets,", len(s1), "set-1 results")

rows = []
for n_done, s in enumerate(s1):
    if n_done % 200 == 0: print(n_done, "checked,", len(rows), "saved so far")
    code, side = s["ticker"].split("-")[1], s["ticker"].split("-")[3]
    T = int(pd.Timestamp(s["close_time"]).timestamp())
    tk = f"KXATPMATCH-{code}-{side}"; flip = tk not in match
    if flip:
        tk = next((t for t in match if t.startswith(f"KXATPMATCH-{code}-")), None)
        if not tk: continue
    c = cdl(tk, T - 4*3600, T + 600)
    if not c: continue
    ts = np.array([x["end_period_ts"] for x in c]); vol = np.array([float(x.get("volume_fp") or 0) for x in c])
    bid = np.array([f(x, "yes_bid") for x in c]); ask = np.array([f(x, "yes_ask") for x in c])
    j = int(np.argmax(ts >= T + 60))
    if ts[j] < T + 60: continue
    rv = pd.Series(vol).rolling(10, min_periods=1).sum().values
    i = int(np.argmax(rv > 0.25 * rv.max())); pre = np.nanmedian(((bid + ask) / 2)[max(0, i-20):max(1, i-2)])
    won = match[tk]["result"] == ("no" if flip else "yes")
    b, a = (1 - ask[j], 1 - bid[j]) if flip else (bid[j], ask[j])
    rows.append(dict(code=code, player=s.get("yes_sub_title"), bid=b, ask=a, won=won,
                     pre=1 - pre if flip else pre, lag=ts[j] - T, start=int(ts[i]), t_end=T))
pd.DataFrame(rows).to_csv("set1_all.csv", index=False); print(len(rows), "matches saved")

7350 match markets, 2803 set-1 results
0 checked, 0 saved so far
200 checked, 136 saved so far
400 checked, 264 saved so far
600 checked, 394 saved so far


/tmp/ipykernel_1176/1126469678.py:38: RuntimeWarning: All-NaN slice encountered
  i = int(np.argmax(rv > 0.25 * rv.max())); pre = np.nanmedian(((bid + ask) / 2)[max(0, i-20):max(1, i-2)])


800 checked, 574 saved so far
1000 checked, 719 saved so far
1200 checked, 864 saved so far
1400 checked, 999 saved so far
1600 checked, 1137 saved so far
1800 checked, 1286 saved so far
2000 checked, 1422 saved so far
2200 checked, 1569 saved so far
2400 checked, 1715 saved so far
2600 checked, 1845 saved so far
2800 checked, 1977 saved so far
1978 matches saved


In [12]:
tk = next(t for t in match if "26MAY30CERLAN" in t)
c = get(f"/historical/markets/{tk}/candlesticks", start_ts=0, end_ts=2_000_000_000, period_interval=60).get("candlesticks", [])
print(len(c)); print(c[:2])

0
[]


In [13]:
s = next(x for x in s1 if "26MAY30CERLAN" in x["ticker"])
T = int(pd.Timestamp(s["close_time"]).timestamp())
for path in [f"/series/KXATPMATCH/markets/{tk}/candlesticks", f"/historical/markets/{tk}/candlesticks"]:
    r = get(path, start_ts=T - 3600, end_ts=T + 600, period_interval=1)
    print(path, "| keys:", list(r.keys()), "| candles:", len(r.get("candlesticks") or []))
    print((r.get("candlesticks") or [r])[-1])
    print()

/series/KXATPMATCH/markets/KXATPMATCH-26MAY30CERLAN-LAN/candlesticks | keys: ['error'] | candles: 0
{'error': {'code': 'not_found', 'message': 'not found'}}

/historical/markets/KXATPMATCH-26MAY30CERLAN-LAN/candlesticks | keys: ['candlesticks', 'ticker'] | candles: 70
{'end_period_ts': 1780148100, 'open_interest': '783025.76', 'price': {'close': '0.2000', 'high': '0.2300', 'low': '0.1900', 'mean': '0.2131', 'open': '0.2300', 'previous': '0.2400'}, 'volume': '4314.31', 'yes_ask': {'close': '0.2000', 'high': '0.2300', 'low': '0.2000', 'open': '0.2300'}, 'yes_bid': {'close': '0.1900', 'high': '0.2200', 'low': '0.1900', 'open': '0.2200'}}



In [14]:
def f(c, k):
    v = c.get(k) or {}
    v = v.get("close_dollars", v.get("close"))
    return float(v) if v is not None else np.nan
vf = lambda x: float(x.get("volume_fp") or x.get("volume") or 0)

old = pd.read_csv("set1_all.csv")
redo = set(old[old.ask.isna()].code)
todo = [x for x in s1 if x["ticker"].split("-")[1] in redo]
print(len(todo), "matches to re-fetch")

rows = []
for n_done, s in enumerate(todo):
    if n_done % 200 == 0: print(n_done, "checked,", len(rows), "saved so far")
    code, side = s["ticker"].split("-")[1], s["ticker"].split("-")[3]
    T = int(pd.Timestamp(s["close_time"]).timestamp())
    tk = f"KXATPMATCH-{code}-{side}"; flip = tk not in match
    if flip:
        tk = next((t for t in match if t.startswith(f"KXATPMATCH-{code}-")), None)
        if not tk: continue
    c = cdl(tk, T - 4*3600, T + 600)
    if not c: continue
    ts = np.array([x["end_period_ts"] for x in c]); vol = np.array([vf(x) for x in c])
    bid = np.array([f(x, "yes_bid") for x in c]); ask = np.array([f(x, "yes_ask") for x in c])
    j = int(np.argmax(ts >= T + 60))
    if ts[j] < T + 60: continue
    rv = pd.Series(vol).rolling(10, min_periods=1).sum().values
    i = int(np.argmax(rv > 0.25 * rv.max())); pre = np.nanmedian(((bid + ask) / 2)[max(0, i-20):max(1, i-2)])
    won = match[tk]["result"] == ("no" if flip else "yes")
    b, a = (1 - ask[j], 1 - bid[j]) if flip else (bid[j], ask[j])
    rows.append(dict(code=code, player=s.get("yes_sub_title"), bid=b, ask=a, won=won,
                     pre=1 - pre if flip else pre, lag=ts[j] - T, start=int(ts[i]), t_end=T))
pd.DataFrame(rows).to_csv("set1_hist.csv", index=False); print(len(rows), "matches saved")


1486 matches to re-fetch
0 checked, 0 saved so far
200 checked, 170 saved so far
400 checked, 330 saved so far
600 checked, 478 saved so far
800 checked, 634 saved so far
1000 checked, 787 saved so far
1200 checked, 937 saved so far
1400 checked, 1090 saved so far
1153 matches saved


In [16]:
pd.set_option("display.max_rows", None)
print(s[s.title.str.contains("NBA", case=False)][["ticker", "title"]])

AttributeError: 'dict' object has no attribute 'title'

In [17]:
pd.set_option("display.max_rows", None)
series = pd.json_normalize(get("/series", category="Sports")["series"])
print(series[series.title.str.contains("NBA", case=False)][["ticker", "title"]])

                      ticker                                          title
46          KXNBAALLSTARGAME                       NBA All-Star Game Winner
74         KXNBAFINALSVIEWER                          NBA finals viewership
81               KXNBADRAFT7                         NBA Draft Seventh Pick
91                   KXNBA2D                       NBA Player Double Double
107               KXWNBACCUP                 WNBA Commissioner's Cup Winner
146         KXNBA1HTEAMTOTAL                        NBA 1st Half Team Total
193                   KXWNBA                              WNBA Championship
222       KXNBAROOKIE1STTEAM                        NBA All-Rookie 1st Team
249                KXNBAGAME                                       NBA Game
251           KXNBAPTSLEADER                         NBA Game Points Leader
263               KXWNBADPOY              WNBA Defensive Player of the Year
327                KXNBASEED                               NBA Playoff Seed
365         

In [18]:
import numpy as np, requests, time
def f(c, k):
    v = c.get(k) or {}; v = v.get("close_dollars", v.get("close"))
    return float(v) if v is not None else np.nan
def cdl_s(series, tk, a, b):
    for path in [f"/series/{series}/markets/{tk}/candlesticks", f"/historical/markets/{tk}/candlesticks"]:
        c = get(path, start_ts=a, end_ts=b, period_interval=1).get("candlesticks")
        if c: return c
    return []

nba = all_mk("KXNBAGAME", n=10000) + all_mk("KXNBAGAME", hist=True, n=10000)
t0, t1 = pd.Timestamp("2025-10-20", tz="UTC"), pd.Timestamp("2026-06-30", tz="UTC")
nba = [m for m in nba if t0 <= pd.Timestamp(m["close_time"]) <= t1 and m.get("result") in ("yes", "no")]
one = {}
for m in nba: one.setdefault(m.get("event_ticker", m["ticker"].rsplit("-", 1)[0]), m)
print(len(nba), "markets,", len(one), "games")
pd.json_normalize(list(one.values()))[["ticker", "title", "yes_sub_title", "result", "close_time"]].to_csv("nba_markets.csv", index=False)

rows = []
for k, m in enumerate(one.values()):
    if k % 200 == 0: print(k, "games done")
    T = int(pd.Timestamp(m["close_time"]).timestamp())
    for x in cdl_s("KXNBAGAME", m["ticker"], T - 5*3600, T):
        b, a = f(x, "yes_bid"), f(x, "yes_ask")
        if not (np.isnan(b) and np.isnan(a)): rows.append((m["ticker"], x["end_period_ts"], b, a))
pd.DataFrame(rows, columns=["ticker", "ts", "bid", "ask"]).to_csv("nba_prices.csv", index=False)
print(len(rows), "price rows saved")

2634 markets, 1317 games
0 games done
200 games done
400 games done
600 games done
800 games done
1000 games done
1200 games done
293930 price rows saved


In [19]:
E = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"
games, plays = [], []
for d in pd.date_range("2025-10-21", "2026-06-25"):
    try: evs = requests.get(f"{E}/scoreboard", params={"dates": d.strftime("%Y%m%d")}, timeout=20).json().get("events", [])
    except Exception: continue
    for e in evs:
        try: j = requests.get(f"{E}/summary", params={"event": e["id"]}, timeout=20).json()
        except Exception: continue
        t = {c["homeAway"]: c for c in j["header"]["competitions"][0]["competitors"]}
        games.append(dict(gid=e["id"], date=d.date(), home=t["home"]["team"]["abbreviation"], away=t["away"]["team"]["abbreviation"],
                          home_id=t["home"]["team"]["id"], away_id=t["away"]["team"]["id"],
                          home_pts=t["home"].get("score"), away_pts=t["away"].get("score")))
        for p in j.get("plays", []):
            if p.get("shootingPlay") or p.get("scoringPlay"):
                plays.append((e["id"], p.get("wallclock"), p["period"]["number"], p["clock"]["displayValue"],
                              p.get("homeScore"), p.get("awayScore"), (p.get("team") or {}).get("id"),
                              "three point" in p.get("text", "").lower(), bool(p.get("scoringPlay"))))
    if d.day == 1: print(d.date(), len(games), "games so far")
pd.DataFrame(games).to_csv("nba_games.csv", index=False)
pd.DataFrame(plays, columns=["gid", "wallclock", "period", "clock", "home", "away", "team_id", "is3", "made"]).to_csv("nba_plays.csv", index=False)
print(len(games), "games,", len(plays), "plays saved")

2025-11-01 86 games so far
2025-12-01 308 games so far
2026-01-01 501 games so far
2026-02-01 743 games so far
2026-03-01 914 games so far
2026-04-01 1151 games so far
2026-05-01 1289 games so far
2026-06-01 1324 games so far
1329 games, 298428 plays saved
